# Chẩn đoán Controller A–E — offline / CPU

Chỉ cần Secret `GITHUB_TOKEN`, quyền mount Drive và artifact P0–P5 đã có. Chọn **CPU runtime** rồi **Runtime → Run all**. Không tải UDIS-D, không chạy CUDA sweep, không thay default policy và không triển khai v5.

Notebook tự tìm full result/checkpoint trên Drive. Nếu chỉ còn ZIP `full-1382a8b5cae5-68c68d47bc6c42ad93cc2974fcd59193.zip`, upload vào `MyDrive/controller-score-results/`; không cần ảnh hoặc ZIP dataset 1.95 GB. Source diagnostics mới phải có trên GitHub trước khi clone.


In [ ]:
# CONFIG — the only cell to edit. Never put a token here.
CONFIG = {
    "REPO": "nh-atuan/crossstage-stitch",
    "REF": "de/optimize-controller",
    "DRIVE_ROOT": "/content/drive/MyDrive/controller-score-work",
    "OUTPUT_ROOT": "/content/drive/MyDrive/controller-diagnostics-AE",
    "SEED": 20261101,
    "ARTIFACT_PATH": "",  # Optional ZIP/folder only if automatic discovery cannot find evidence.
}


## 1. Mount Drive

Đồng ý prompt tiêu chuẩn của Google để đọc kết quả cũ và lưu checkpoint. CPU runtime đủ; nếu đang dùng T4 thì notebook vẫn chỉ chạy CPU, không khởi chạy phép đo CUDA.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")


## 2. Clone private repo và khóa source

Secret `GITHUB_TOKEN` dùng Fine-grained PAT với Contents: Read-only cho đúng repo và Notebook access ON. Không dán token vào CONFIG. Branch được fetch một lần ở đầu Run all, rồi full SHA được ghi trong metadata. Git dùng ASKPASS tạm; token chỉ vào môi trường tiến trình con, không vào URL/remote/log/ZIP.


In [ ]:
import re
import shutil
import tempfile


def checkout_private(repository, ref, workspace, get_secret):
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository) or repository == "OWNER/REPO":
        raise RuntimeError("Điền CONFIG.REPO theo dạng OWNER/REPO.")
    if not ref or ref.startswith("-") or any(c in ref for c in "\r\n"):
        raise RuntimeError("Điền CONFIG.REF bằng branch hoặc commit hợp lệ đã push.")
    checkout = Path(workspace) / "controller-score-source"
    lock = checkout / ".git/score-checkout.json"
    existing = lock.exists()
    if existing:
        stored = json.loads(lock.read_text())
        actual = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(checkout), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
        if stored["repository"] != repository or stored["revision"] != actual or dirty:
            raise RuntimeError("Checkout đã đổi repo/HEAD hoặc có sửa tracked files. Giữ checkpoint trên Drive và dùng checkout sạch; không ghi đè thay đổi.")
        if re.fullmatch(r"[0-9a-f]{40}", ref) and actual == ref:
            return checkout, actual
    elif checkout.exists():
        raise RuntimeError("Checkout dở dang. Xóa riêng /content/controller-score-source trong Files rồi Run all; giữ checkpoint trên Drive.")
    try:
        token = get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Không đọc được GITHUB_TOKEN. Mở Secrets, thêm token và bật Notebook access; kiểm quyền repository.") from None
    if not isinstance(token, str) or not token.strip():
        token = None
        raise RuntimeError("GITHUB_TOKEN trống. Tạo Fine-grained PAT Contents: Read-only cho đúng repo trong Colab Secrets.")
    child_env = {k:v for k,v in os.environ.items() if not k.startswith("GIT_") and k not in ("GITHUB_TOKEN", "GH_TOKEN")}
    try:
        with tempfile.TemporaryDirectory(prefix="score-git-auth-") as auth:
            askpass = Path(auth) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['SCORE_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            child_env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0", SCORE_GITHUB_TOKEN=token)
            args = ["git", "-c", "credential.helper=", "-c", "core.askPass="+str(askpass), "-c", "http.extraHeader="]
            commands = [] if existing else [args+["init", str(checkout)],
                args+["-C", str(checkout), "remote", "add", "origin", "https://github.com/"+repository+".git"]]
            commands += [args+["-C", str(checkout), "fetch", "--depth=1", "origin", ref],
                args+["-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"]]
            for command in commands:
                completed = subprocess.run(command, env=child_env, capture_output=True, text=True)
                if completed.returncode:
                    # Suppress all network output; no credential-bearing output reaches a cell/log.
                    raise RuntimeError("Clone/checkout thất bại. Kiểm OWNER/REPO, ref đã push, Contents: Read-only và approval của tổ chức; không đổi sang token ghi hoặc URL chứa token.")
    finally:
        token = None
        child_env.pop("SCORE_GITHUB_TOKEN", None)
        child_env.pop("GIT_ASKPASS", None)
    revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Không khóa được full commit SHA")
    lock.write_text(json.dumps({"repository": repository, "ref": ref, "revision": revision}))
    return checkout, revision


from google.colab import userdata
SOURCE, REVISION = checkout_private(CONFIG["REPO"], CONFIG["REF"], "/content", userdata.get)
os.chdir(SOURCE)
if not (SOURCE/"tools/controller/diagnostics/runner.py").is_file():
    raise RuntimeError("Ref chưa chứa diagnostics A–E. Push commit đã bàn giao hoặc chọn branch/commit có tools/controller/diagnostics/runner.py.")
print("LOCKED COMMIT:", REVISION)

import importlib
for _name in list(sys.modules):
    if _name == "tools" or _name.startswith("tools."):
        del sys.modules[_name]
importlib.invalidate_caches()
sys.path.insert(0, str(SOURCE))


## 3. Tìm artifact đã có

Ưu tiên dữ liệu trong repo, thư mục work/results trên Drive, ZIP P0–P5, rồi ARTIFACT_PATH. Không dùng đường dẫn ảnh cũ để giả định ảnh tồn tại. Nếu thiếu dữ liệu, A–E vẫn được chạy theo khả năng và xuất trạng thái BLOCKED/PARTIAL cùng danh sách file cần bổ sung.


In [ ]:
from tools.controller.diagnostics.discovery import locate
FOUND = locate(SOURCE, Path(CONFIG["DRIVE_ROOT"]), CONFIG["ARTIFACT_PATH"])
print(json.dumps(FOUND, indent=2, ensure_ascii=False))


## 4. Dependencies CPU

Dùng XGBoost CPU 3.2.0 và requirements báo cáo hiện có. Không cài quantum SDK, không yêu cầu nvcc hoặc GPU. Model chẩn đoán mới chỉ ghi vào output thí nghiệm.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbformat", "nbclient", "ipykernel",
    "-r", "tools/controller/requirements-learn.txt", "-r", "tools/controller/requirements-report.txt"], check=True)


## 5. Tests và data audit

Kiểm fixture CPU khép kín A–E, split/group leakage, metric, replay validity, model/feature mismatch, notebook và ZIP. Sau đó audit dữ liệu thật trước mọi thí nghiệm. Nếu test lỗi thì dừng; nếu thiếu artifact thì xuất báo cáo thiếu dữ liệu, không tự chạy sweep.


In [ ]:
_validation = Path(CONFIG["OUTPUT_ROOT"]) / "validation"
_validation.mkdir(parents=True, exist_ok=True)
_test_log = _validation / "cpu-tests.log"
with _test_log.open("w") as _log:
    _returncode = subprocess.run([sys.executable, "-m", "pytest",
        "tests/python/test_controller_diagnostics.py", "tests/python/test_diagnostics_notebook.py",
        "-q", "-p", "no:cacheprovider"], cwd=SOURCE, stdout=_log, stderr=subprocess.STDOUT).returncode
print(_test_log.read_text()[-10000:])
if _returncode:
    raise RuntimeError("CPU tests failed; xem " + str(_test_log))
from tools.controller.diagnostics.runner import start, run_experiment, finish
CTX = start(CONFIG, repo=SOURCE, found=FOUND)
import shutil
shutil.copy2(_test_log, CTX.output / "cpu-tests.log")
print((CTX.output / "data-inventory.csv").read_text())


## A. Pre-model: ranking, filter hay scoring?

Đo AUC/AP/Brier và confusion theo K/scale; cheap-feasible recall, top-k feasible/near-Oracle, regret và phân biệt Oracle bị pre-filter hay score loại. OOF là nguồn chính; model cuối trên calibration được ghi in-sample riêng. Không dùng selected routes thay full evaluation grid.


In [ ]:
A = run_experiment(CTX, "A")


## B. Probability calibration

So raw sigmoid với probability calibration đã lưu trên đúng outer models. Reliability có 10 bin đều và CI bootstrap theo component. Post invalid/missing được giữ mask; route evaluation chỉ là thống kê conditional trên dữ liệu đã xem. Gate 0.65/0.85 được so diagnostic, không chọn default từ chúng.


In [ ]:
B = run_experiment(CTX, "B")


## C. Perfect pre / perfect latency — chỉ offline

Thay lần lượt p_pre bằng QoR truth, L_hat bằng fixed latency thật và cả hai; giữ post-gate thật. Không xuất nhãn Oracle vào inference. Thiếu transition timing thì báo các scenario ESTIMATED với assumptions, không gọi là GPU E2E hoặc timing bounds đã chứng minh.


In [ ]:
C = run_experiment(CTX, "C")


## D. Perfect post-gate

Giữ initial route; oracle post vẫn phải qua self-check và recorded-feature validity. Case study cố định 001855/009558/002508/009333 gồm p_pre/p_post/score, QoR margins, decision và transition. Nhánh mới chưa đo retry/S6 có cost UNKNOWN, không lấy số từ pair khác.


In [ ]:
D = run_experiment(CTX, "D")


## E. Learning curve theo group độc lập

Giữ outer evaluation groups, capacity 64 cây depth 2 và ba sampling seed. Các cỡ 40/80/120/160 được cap theo pool outer-training khả thi, thêm full pool. Với 195 pair/5 folds, full pool tối đa 156 group; không fit 195 rồi gọi chính chúng là held-out. Booster-fit và probability-calibration luôn tách group; P0–P5 evaluation không đi vào train.

Mỗi fit có model/split/metric checkpoint. Sampling seeds được gộp theo held-out pair trước component bootstrap; không coi configs hoặc seed là ảnh độc lập. Không chọn model capacity/threshold bằng evaluation đã xem.


In [ ]:
E = run_experiment(CTX, "E")


## 6. Tổng hợp và ZIP

Bảng dưới in A–E COMPLETED/PARTIAL/BLOCKED/FAILED, số pair và kết luận cả kết quả âm/INCONCLUSIVE. conclusions.md dẫn về CSV/JSON cụ thể. ZIP chứa protocol/source/model/input hashes, folds/models chẩn đoán, logs, bảng và figure PNG/PDF. Checkpoint/output nằm trên Drive; không sửa policy mặc định.


In [ ]:
RESULT_ZIP = finish(CTX)
print((CTX.output / "experiment-status.csv").read_text())
print((CTX.output / "conclusions.md").read_text())


## 7. Tải và gửi lại

Gửi ZIP kết quả nguyên vẹn cùng LOCKED COMMIT trong chat để phân tích tiếp. Bản ZIP cũng được giữ trong OUTPUT_ROOT trên Drive. Nếu file lớn, lấy trực tiếp từ Drive thay vì tải qua trình duyệt; không cần gửi dataset hoặc token.


In [ ]:
from google.colab import files
print("RESULT ZIP:", RESULT_ZIP)
print("LOCKED COMMIT:", REVISION)
if RESULT_ZIP.stat().st_size <= 250 * 1024 * 1024:
    files.download(str(RESULT_ZIP))
else:
    print("ZIP lớn; tải trực tiếp từ Drive tại đường dẫn trên.")
